# Taller: GAN con Quick, Draw! — categoría **apple** (manzana)

En este taller construimos una **GAN** (Red Generativa Antagónica) con **dos redes neuronales**:

| Red | Rol | Archivo donde se crea |
|---|---|---|
| **Generadora (G)** | Convierte ruido aleatorio en un dibujo de manzana | `src/models/generator.py` |
| **Discriminadora (D)** | Decide si un dibujo es real o falso | `src/models/discriminator.py` |

**Contenido:** 1. Datos · 2. Generador · 3. Discriminador · 4. Funciones de pérdida · 5. Entrenamiento · 6. Valores de las redes · 7. Resultado final.

> El código real vive en `src/` (con comentarios línea a línea). Este notebook lo usa paso a paso.

In [ ]:
import sys
from pathlib import Path

# Permite importar el paquete `src` desde la carpeta notebooks/.
sys.path.insert(0, str(Path.cwd().parent))

import matplotlib.pyplot as plt
import torch

from src.config import Config, get_device
from src.utils.logger import setup_logging
from src.utils.reproducibility import set_seed

setup_logging()
config = Config()  # <- TODOS los hiperparámetros están en src/config.py
device = get_device()
set_seed(config.seed)

print(f"Categoría usada : {config.category}")
print(f"Dispositivo     : {device}")
print(config)

## 1. Datos: descargar UNA categoría de Quick, Draw!

Dataset oficial: <https://github.com/googlecreativelab/quickdraw-dataset>.
Usamos el formato *numpy bitmap*: imágenes de **28×28** píxeles en escala de grises.
Las normalizamos al rango **[-1, 1]** porque la última capa del Generador es `Tanh`.

In [ ]:
from src.data.dataset import QuickDrawDataset, create_dataloader
from src.data.downloader import download_category
from src.utils.visualization import plot_image_grid

download_category(config.category, config.data_path, config.img_size)
dataset = QuickDrawDataset(config.data_path, config.num_samples, config.img_size)
dataloader = create_dataloader(
    dataset, config.batch_size, config.seed, pin_memory=device.type == "cuda"
)

print(f"Imágenes de entrenamiento: {len(dataset)} | lotes por época: {len(dataloader)}")
plot_image_grid(dataset.images[:64], f"Dibujos REALES - Categoría: {config.category}")
plt.show()

## 2. Red neuronal GENERADORA (G)

Recibe un vector de ruido `z` de **100** números y lo transforma en una imagen de 28×28 mediante
capas `ConvTranspose2d` (convoluciones que *amplían*). Mira `src/models/generator.py`:
la creación ocurre en la función `build_generator` (marcada con `>>> AQUÍ SE CREA LA RED NEURONAL GENERADORA <<<`).

In [ ]:
from src.models.generator import build_generator
from src.utils.inspect_models import format_model_report

# >>> AQUÍ SE CREA LA RED GENERADORA <<<
generator = build_generator(config, device)

z_ejemplo = torch.randn(2, config.latent_dim, device=device)
print(format_model_report("RED GENERADORA (sin entrenar)", generator, z_ejemplo))

**Cómo leer la tabla:**
- *Forma de salida* `[2, 256, 7, 7]` = (lote, canales, alto, ancho). Fíjate cómo el tamaño crece 1×1 → 7×7 → 14×14 → 28×28.
- *Parámetros* = pesos que se aprenden. En `ConvTranspose2d` son `entrada × salida × kernel × kernel`
  (p. ej. `100 × 256 × 7 × 7 = 1 254 400`).
- La media ≈ 0 y la desviación ≈ 0.02 de los pesos vienen de la inicialización DCGAN `Normal(0, 0.02)`.

## 3. Red neuronal DISCRIMINADORA (D)

Recibe una imagen 28×28 y devuelve **un solo número** (*logit*): alto = «parece real», bajo = «parece falsa».
Usa `Conv2d` (que *reducen*), `LeakyReLU` y `Dropout`. Está en `src/models/discriminator.py`
(`build_discriminator`, marcada con `>>> AQUÍ SE CREA LA RED NEURONAL DISCRIMINADORA <<<`).

In [ ]:
from src.models.discriminator import build_discriminator

# >>> AQUÍ SE CREA LA RED DISCRIMINADORA <<<
discriminator = build_discriminator(config, device)

imagen_ejemplo = torch.zeros(2, 1, config.img_size, config.img_size, device=device)
print(format_model_report("RED DISCRIMINADORA (sin entrenar)", discriminator, imagen_ejemplo))

## 4. Funciones de pérdida

$$\min_G \max_D \; \mathbb{E}_x[\log D(x)] + \mathbb{E}_z[\log(1 - D(G(z)))]$$

| Red | Quiere... | Pérdida |
|---|---|---|
| D | `D(x)→1` y `D(G(z))→0` | $L_D = -\log D(x) - \log(1 - D(G(z)))$ |
| G | `D(G(z))→1` (engañar a D) | $L_G = -\log D(G(z))$ |

Implementadas en `src/training/losses.py`. Comprobemos valores conocidos:

In [ ]:
from src.training.losses import discriminator_loss, generator_loss

casos = {
    "D no distingue (logit=0)": (torch.zeros(8), torch.zeros(8)),
    "D gana (reales=+5, falsas=-5)": (torch.full((8,), 5.0), torch.full((8,), -5.0)),
    "G engaña a D (reales=+5, falsas=+5)": (torch.full((8,), 5.0), torch.full((8,), 5.0)),
}
print(f"{'Situación':<40}{'L_D':>8}{'L_G':>8}")
for nombre, (real_logits, fake_logits) in casos.items():
    ld = discriminator_loss(real_logits, fake_logits).total.item()
    lg = generator_loss(fake_logits).item()
    print(f"{nombre:<40}{ld:>8.3f}{lg:>8.3f}")
print("\nReferencia del equilibrio: L_D = 2*ln2 = 1.386 | L_G = ln2 = 0.693")

## 5. Entrenamiento

En **cada lote** se hacen dos pasos: (1) D aprende a distinguir reales de falsas; (2) G aprende a engañar a D.
Ambas redes tienen su propio optimizador **Adam** (`lr=0.0002`, `beta1=0.5`). Tarda ~1 minuto con GPU.

In [ ]:
from src.training.trainer import GANTrainer

trainer = GANTrainer(generator, discriminator, config, device)
history = trainer.fit(dataloader)
trainer.save_checkpoint()
history.save(config.outputs_dir / "history.json")

## 6. Ver los valores del Generador y del Discriminador (ya entrenados)

Compara los pesos con los de antes de entrenar: la media y la desviación cambian porque las redes **aprendieron**.

In [ ]:
from src.utils.inspect_models import format_output_report

print(format_model_report("RED GENERADORA (entrenada)", generator, z_ejemplo))
print()
print(format_model_report("RED DISCRIMINADORA (entrenada)", discriminator, imagen_ejemplo))
print()
real_batch = dataset.images[: config.sample_grid_size].to(device)
print(format_output_report(generator, discriminator, real_batch, config))

## 7. Resultado de la GAN — categoría: **apple**

### 7.1 Curvas de pérdida y salidas del Discriminador

In [ ]:
from src.utils.visualization import plot_real_vs_fake, plot_training_curves

plot_training_curves(history, config.category)
plt.show()

**Cómo interpretarlas:**
- Si `D(x)` ≈ 0.5 y `D(G(z))` ≈ 0.5, D no distingue (equilibrio ideal). Aquí D va algo por delante
  (`D(x)` entre 0.8 y 0.9, `D(G(z))` entre 0.1 y 0.2) y aun así G produce manzanas reconocibles.
- Las pérdidas **oscilan** (no bajan de forma monótona como en un clasificador): G y D se persiguen mutuamente.
  Esto es normal en una GAN.

### 7.2 Dibujos reales vs. generados

In [ ]:
fake_batch = trainer.generate_samples()
plot_real_vs_fake(real_batch, fake_batch, config.category)
plt.show()

### 7.3 Evolución durante el entrenamiento
Mira `outputs/apple/evolution.gif` (se genera con `python -m src.train`) o las imágenes de `outputs/apple/samples/`:
de ruido puro (época 0) a manzanas reconocibles.
